# Task 2 - Step 1: Train Source-only, DAN, DANN, CDAN (+ DAN λ study)

Every run goes through **one** training loop (`shared/training.py`) with identical initialisation, source
sampling and augmentation, optimiser (AdamW, lr 1e-4, wd 1e-4), epoch budget (≤ 30 epochs of
ceil(largest source-train domain / 8) updates), frozen BatchNorm statistics, and early stopping / checkpoint
selection on **mean source-validation macro-F1** (patience 5). Only the loss differs (`task2/methods/`).

| Config | Method | Target images used? |
|---|---|---|
| `source_only` | CE on 8+8+8 source images | **no** (so it is a valid Task 3 ERM baseline) |
| `dan` | CE + 1.0 · MK-MMD² (512-d feature) | yes, 24 per update, unlabelled |
| `dann` | CE + domain CE through gradient reversal, α(p) standard schedule (manual settings) | yes |
| `cdan` | as DANN, discriminator input vec(f ⊗ p) (manual settings) | yes |
| `dann_disclr10`, `cdan_disclr10` | same, discriminator learning rate ×10 (first stabilisation attempt) | yes |
| `dann_discbn`, `cdan_discbn` | same, BatchNorm in the discriminator hidden layer, manual lr (**main comparison**) | yes |
| `dan_lambda0.1`, `dan_lambda10` | controlled study (λ = 0.1, 10) | yes |

Target images enter as images only; **no Sketch label exists anywhere in this notebook**.

* Existing runs (a `summary.json` in `task2/checkpoints/<run>/`) are **skipped, never overwritten**.
* Run a subset with the env var `TASK2_RUNS=source_only,dan`.
* Expected cost: roughly 10-15 min per run on an RTX 4060 (6 runs).

In [1]:
# ---- Task 2 common header (identical in every Task 2 notebook) ----
import json, os, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Locate the repository root (the folder containing shared/) and make it importable.
REPO = Path.cwd().resolve()
while not (REPO / "shared" / "pacs.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("Run this notebook from inside the PA1 repository")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from shared import pacs, pacs_protocol as proto
from shared.config import load_config

T2 = REPO / "task2"
CFG_DIR = T2 / "configs"
SEED = 6304
# Smoke mode (env TASK2_SMOKE=1): 2 epochs x 5 updates per run, all outputs under _smoke/ folders,
# and notebook 03 uses RANDOM labels instead of loading the real Sketch labels.
SMOKE = os.environ.get("TASK2_SMOKE", "0") == "1"
SUB = "_smoke" if SMOKE else ""
RES = T2 / "results" / SUB
TAB, FIG = RES / "tables", RES / "figures"
DATA_TAB = T2 / "results" / "tables"      # data-preparation tables (same in smoke and real mode)
CKPT = T2 / "checkpoints" / SUB            # git-ignored
CACHE = T2 / "cache" / SUB                 # git-ignored
for p in [TAB, FIG, CKPT, CACHE]:
    p.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Config names (files in task2/configs). The main comparison uses lambda = 1 for DAN.
# DANN/CDAN with the manual's exact settings were unstable (DANN diverged, CDAN collapsed repeatedly). Two fixes
# were tried, both chosen from source-side evidence only (see task2/RUN_LOG.md):
#   1) discriminator lr x10 (2026-09-22, before target evaluation): CDAN stable, DANN diverged at epoch 10;
#   2) BatchNorm in the discriminator hidden layer, manual lr (post-evaluation rerun after root-cause diagnosis).
# Fix 2 passes the pre-set stability criteria for both and is the main DANN/CDAN row; all other runs are reported.
MAIN_RUNS = ["source_only", "dan", "dann_discbn", "cdan_discbn"]
ADAPT_MAIN = ["dan", "dann_discbn", "cdan_discbn"]
AS_SPECIFIED_ADV = ["dann", "cdan"]
DISCLR10_ADV = ["dann_disclr10", "cdan_disclr10"]
ADV_RUNS = ["dann_discbn", "cdan_discbn", "dann_disclr10", "cdan_disclr10", "dann", "cdan"]
STUDY_RUNS = ["dan_lambda0.1", "dan", "dan_lambda10"]          # DAN lambda in {0.1, 1, 10}
ALL_RUNS = MAIN_RUNS + DISCLR10_ADV + AS_SPECIFIED_ADV + ["dan_lambda0.1", "dan_lambda10"]
RUN_NAME = {c: load_config(CFG_DIR, c)["run_name"] for c in ALL_RUNS}
LABEL = {"source_only": "Source-only", "dan": "DAN (λ=1)", "dann_discbn": "DANN (disc BN)", "cdan_discbn": "CDAN (disc BN)",
         "dann_disclr10": "DANN (disc lr×10)",
         "cdan_disclr10": "CDAN (disc lr×10)", "dann": "DANN (as specified)", "cdan": "CDAN (as specified)",
         "dan_lambda0.1": "DAN (λ=0.1)", "dan_lambda10": "DAN (λ=10)"}

plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "legend.fontsize": 9, "savefig.dpi": 150})


def savefig(fig, stem):
    """Every figure is saved as PNG + PDF (never only displayed)."""
    fig.savefig(FIG / f"{stem}.png", bbox_inches="tight")
    fig.savefig(FIG / f"{stem}.pdf", bbox_inches="tight")
    print("saved figure", stem)


print("REPO:", REPO, "| device:", DEVICE, "| smoke:", SMOKE)

REPO: C:\Users\afifh\Desktop\ATML\PA1 | device: cuda | smoke: False


In [2]:
from shared.models import build_model
from shared.training import train
from task2.methods import build_method

source = proto.load_source_data()
target_images = pacs.load_target_images()            # images only - labels are not loaded
print({d: len(source["train"][d][1]) for d in source["train"]}, "| val:", {d: len(source["val"][d][1]) for d in source["val"]},
      "| target images:", len(target_images))

{'photo': 1336, 'art_painting': 1638, 'cartoon': 1875} | val: {'photo': 334, 'art_painting': 410, 'cartoon': 469} | target images: 3929


In [3]:
runs = os.environ.get("TASK2_RUNS", ",".join(ALL_RUNS)).split(",")
SUMMARIES = {}
for name in runs:
    cfg = load_config(CFG_DIR, name)
    if SMOKE:
        cfg["train"]["max_epochs"] = 2
    run_dir = CKPT / cfg["run_name"]
    if (run_dir / "summary.json").exists():
        print(f"skip {name}: finished run already in {run_dir}")
        SUMMARIES[name] = json.loads((run_dir / "summary.json").read_text())
        continue
    print(f"\n=== {name} ({cfg['method']}) ===")
    model = build_model(cfg["model"]["num_classes"], cfg["seed"])
    method = build_method(cfg)
    SUMMARIES[name] = train(model, method, source, cfg, run_dir, DEVICE,
                            target_images=target_images if method.uses_target else None,
                            iters_override=5 if SMOKE else None)
    del model, method
    torch.cuda.empty_cache()

skip source_only: finished run already in C:\Users\afifh\Desktop\ATML\PA1\task2\checkpoints\source_only
skip dan: finished run already in C:\Users\afifh\Desktop\ATML\PA1\task2\checkpoints\dan_lambda1
skip dann_discbn: finished run already in C:\Users\afifh\Desktop\ATML\PA1\task2\checkpoints\dann_discbn
skip cdan_discbn: finished run already in C:\Users\afifh\Desktop\ATML\PA1\task2\checkpoints\cdan_discbn
skip dann_disclr10: finished run already in C:\Users\afifh\Desktop\ATML\PA1\task2\checkpoints\dann_disclr10
skip cdan_disclr10: finished run already in C:\Users\afifh\Desktop\ATML\PA1\task2\checkpoints\cdan_disclr10
skip dann: finished run already in C:\Users\afifh\Desktop\ATML\PA1\task2\checkpoints\dann
skip cdan: finished run already in C:\Users\afifh\Desktop\ATML\PA1\task2\checkpoints\cdan
skip dan_lambda0.1: finished run already in C:\Users\afifh\Desktop\ATML\PA1\task2\checkpoints\dan_lambda0.1
skip dan_lambda10: finished run already in C:\Users\afifh\Desktop\ATML\PA1\task2\checkpo

In [4]:
rows = [{"config": n, "run": s["run_name"], "method": s["method"], "uses_target_images": s["uses_target_images"],
         "best_epoch": s["best_epoch"], "epochs_run": s["epochs_run"], "stopped_early": s["stopped_early"],
         "best_mean_val_macro_f1": s["best_mean_val_macro_f1"], "train_minutes": s["train_seconds"] / 60}
        for n, s in SUMMARIES.items()]
TRAIN_SUMMARY = pd.DataFrame(rows)
TRAIN_SUMMARY.to_csv(TAB / "task2_training_summary.csv", index=False)
TRAIN_SUMMARY.round(4)

,config,run,method,uses_target_images,best_epoch,epochs_run,stopped_early,best_mean_val_macro_f1,train_minutes
0,source_only,source_only,source_only,False,6,11,True,0.9317,2.2367
1,dan,dan_lambda1,dan,True,5,10,True,0.9428,4.0938
2,dann_discbn,dann_discbn,dann,True,6,11,True,0.9443,4.5892
3,cdan_discbn,cdan_discbn,cdan,True,4,9,True,0.9391,3.8044
4,dann_disclr10,dann_disclr10,dann,True,8,13,True,0.9484,5.5970
5,cdan_disclr10,cdan_disclr10,cdan,True,6,11,True,0.9428,4.7766
6,dann,dann,dann,True,1,6,True,0.4418,2.5019
7,cdan,cdan,cdan,True,2,7,True,0.9026,3.2318
8,dan_lambda0.1,dan_lambda0.1,dan,True,4,9,True,0.9398,4.6529
9,dan_lambda10,dan_lambda10,dan,True,1,6,True,0.0507,2.5700
